# Suite 1: MacroPlan-Bench — Standalone Outline Planning Benchmark

Đánh giá **chất lượng dàn ý (Macro Event Graph & Chapter Blueprint)** trước khi sinh văn bản prose:
- **Trục 1 — Premise Grounding**: `S_cov` + Judge `outline_relevance`
- **Trục 2 — Causal Integrity & Progression**: `S_causal` (DAG + DeBERTa-v3 NLI), `S_prog`, `S_comp`
- **Trục 3 — CONCOCT Concreteness & Pacing Balance**: `mean_concreteness`, `pacing_balance_score`, `sparse_event_ratio`, `inter_event_diversity`, `plan_critic_pass_rate`
- **Trục 4 — Narrative Appeal (DOC + HANNA)**: 4-dim rubric + Blind Pairwise A/B (swapped order)
- **Trục 5 — Predictive Utility & Cost**: `planning_tokens` + tùy chọn VR-CLI `I_pi`
- **Phần 3 (Tùy chọn) — Meta-Evaluation**: Perturbation Sanity Check tự kiểm chứng độ nhạy metric.

> Chạy qua subprocess sạch (`scripts/benchmark_outline.py`) với `VLLM_WORKER_MULTIPROC_METHOD=spawn` nên không bao giờ xung đột VRAM giữa NLI/MiniLM và vLLM Judge.

In [ ]:
# 1. Mount Google Drive & Sync Repository
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/MA-GRPO')
REPO_URL = 'github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN') or os.environ.get('GH_TOKEN', '')
except Exception:
    GH_TOKEN = os.environ.get('GH_TOKEN', '')

clone_url = f'https://{GH_TOKEN}@{REPO_URL}' if GH_TOKEN else f'https://{REPO_URL}'
if not REPO_DIR.exists():
    !git clone {clone_url} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 2. Install Lightweight Dependencies (NLI, MiniLM, OpenAI client; vLLM only if using local_vllm)
!pip install -q transformers sentence-transformers accelerate openai matplotlib pandas

In [ ]:
# 3. Configure Inputs & Judge Backend
# Hỗ trợ đồng thời nhiều file dàn ý để so sánh (vd: shared_macro_plans.json, grpo_train_condition_d.jsonl, track2_generated.jsonl)
INPUT_FILES = [
    '/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation/shared_macro_plans.json',
]
GROUP_LABELS = [
    'condition_d_32b',
]

OUTPUT_DIR = '/content/drive/MyDrive/MA-GRPO-Benchmarks/suite1_outline_bench'

# Judge backend: 'none' (chỉ metric tự động + NLI + CONCOCT), 'local_vllm' (32B-AWQ trên A100), hoặc 'api'
JUDGE_BACKEND = 'none'  # Đổi thành 'local_vllm' hoặc 'api' khi muốn chấm DOC+HANNA Judge
JUDGE_MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'
PAIRWISE_GROUPS = None  # Ví dụ: ('condition_d_32b', 'condition_b_flat') nếu có >= 2 nhóm chung premise_id
RUN_SANITY_CHECK = True
AUTO_DISCONNECT = False

# Nếu dùng JUDGE_BACKEND == 'local_vllm', tự động cài vllm nếu chưa có
if JUDGE_BACKEND == 'local_vllm':
    import importlib.util
    if importlib.util.find_spec('vllm') is None:
        !pip install -q vllm

In [ ]:
# 4. Execute Suite 1 (MacroPlan-Bench) via Clean Subprocess
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/benchmark_outline.py',
    '--inputs', *INPUT_FILES,
    '--labels', *GROUP_LABELS,
    '--output-dir', OUTPUT_DIR,
    '--judge-backend', JUDGE_BACKEND,
    '--judge-model', JUDGE_MODEL,
]
if PAIRWISE_GROUPS:
    cmd.extend(['--pairwise-groups', PAIRWISE_GROUPS[0], PAIRWISE_GROUPS[1]])
if RUN_SANITY_CHECK:
    cmd.append('--run-sanity-check')

env = os.environ.copy()
env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
env['VLLM_LOGGING_LEVEL'] = 'WARNING'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'benchmark_outline.py exited with code {ret}')

In [ ]:
# 5. Display Summary Table, Markdown Report & 4-Panel Visualization
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

out_path = Path(OUTPUT_DIR)
df = pd.read_csv(out_path / 'outline_benchmark_summary.csv')
display(df)
display(Markdown((out_path / 'outline_benchmark_report.md').read_text(encoding='utf-8')))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
panels = [
    ('S_causal', 'Causal Coherence (S_causal)'),
    ('pacing_balance_score', 'CONCOCT Pacing Balance'),
    ('mean_concreteness', 'Mean Event Concreteness'),
    ('inter_event_diversity', 'Inter-Event Diversity'),
]
for ax, (col, title) in zip(axes, panels):
    vals = df[col].fillna(0)
    ax.bar(df['group'], vals, color='#2b6cb0')
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(out_path / 'outline_benchmark_chart.png', dpi=150)
plt.show()

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()